# Lab 12 — Cortex Agent: Multi-Tool Orchestration

Cortex Agent orchestrates multiple tools (Cortex Search, Cortex Analyst, APIs) to answer complex queries.

| Item | Detail |
|---|---|
| **Feature** | Cortex Agent REST API |
| **Tools** | Cortex Search + Cortex Analyst + LLM |
| **Exam Domain** | 2.0 Gen AI & LLM Functions |
| **You'll learn** | Agent architecture, tool configuration, multi-step reasoning |


## Step 1 — Environment Setup

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Key Concepts

### Agent Architecture
A Cortex Agent:
1. Receives a natural language question
2. Decides which **tool(s)** to invoke
3. Executes tools and synthesizes results
4. Returns a coherent answer

### Available Tools
| Tool Type | What It Does |
|---|---|
| `cortex_search` | Semantic document retrieval (RAG) |
| `cortex_analyst` | Text-to-SQL on structured data |
| `function` | Custom SQL/Python functions |

### Agent vs. Search vs. Analyst
| Feature | Search | Analyst | Agent |
|---|---|---|---|
| Input | Text query | Natural language question | Natural language question |
| Output | Relevant documents | SQL + results | Synthesized answer |
| Multi-tool | No | No | **Yes** |


## Step 2 — Agent REST API Structure

Agents are accessed via the `/api/v2/cortex/agent:run` REST endpoint.

```json
POST /api/v2/cortex/agent:run
{
  "model": "claude-sonnet-4-5",
  "tools": [
    {"tool_spec": {"type": "cortex_search", "name": "wiki_search"}},
    {"tool_spec": {"type": "cortex_analyst", "name": "sales_analyst",
      "semantic_model_file": "@semantic_models/sales_semantic_model.yaml"}}
  ],
  "messages": [{"role": "user", "content": "Compare Q1 revenue to Q2"}]
}
```

> **Note:** Agents are called via REST API or Streamlit, not direct SQL.


## Step 3 — Verify Prerequisites

An Agent needs tools to be set up first. Let's verify ours are ready.

In [ ]:
SHOW CORTEX SEARCH SERVICES;

In [ ]:
LIST @semantic_models;

## Step 4 — Building an Agent in Streamlit

The typical pattern for a Cortex Agent app:

```python
import streamlit as st
from snowflake.core import Root

root = Root(st.connection("snowflake").session())
agent = root.databases["GENAI_LEARNING"].schemas["PUBLIC"].cortex_agent(
    model="claude-sonnet-4-5",
    tools=[
        {"tool_spec": {"type": "cortex_search_service",
         "name": "GENAI_LEARNING.PUBLIC.WIKI_SEARCH"}},
        {"tool_spec": {"type": "cortex_analyst_tool",
         "semantic_model_file": "@semantic_models/sales_semantic_model.yaml"}}
    ]
)
response = agent.run(messages=[{"role": "user", "content": user_question}])
```


## Key Takeaways

- **Cortex Agent** orchestrates multiple tools (Search + Analyst + functions).
- Agents decide **which tool** to use based on the question — no routing code needed.
- Accessed via REST API (`/api/v2/cortex/agent:run`) or Python SDK.
- Agent = Search (unstructured) + Analyst (structured) + reasoning.
- Build the UX in Streamlit in Snowflake for a complete AI application.
